# Fine-Tuning BERT for POS Tagging & Chunking

In [13]:
!pip install transformers datasets seqeval evaluate accelerate nltk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [19]:
# Import required libraries
import nltk
from nltk.corpus import conll2000
from datasets import Dataset, DatasetDict

# Download dataset
nltk.download('conll2000')

# Function to convert dataset into required format
def parse_data(tagged_sents):
    data = []
    for sent in tagged_sents:
        tokens, pos_tags, chunk_tags = [], [], []

        # Extract word, POS tag, and chunk tag
        for word, pos, chunk in sent:
            tokens.append(word)
            pos_tags.append(pos)
            chunk_tags.append(chunk)

        data.append({
            "tokens": tokens,
            "pos_tags": pos_tags,
            "chunk_tags": chunk_tags
        })
    return data

[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!


In [20]:
#load dataset

train_data= parse_data(conll2000.iob_sents('train.txt'))
test_data= parse_data(conll2000.iob_sents('test.txt'))

# Split training into train + validation

split = int(0.9 * len(train_data))

dataset = DatasetDict({
    "train": Dataset.from_list(train_data[:split]),
    "validation": Dataset.from_list(train_data[split:]),
    "test": Dataset.from_list(test_data)

})

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


# Dataset Size Reduction for Faster Training

In [21]:
# Reduce dataset size for faster execution
dataset["train"] = dataset["train"].select(range(800))
dataset["validation"] = dataset["validation"].select(range(200))
dataset["test"] = dataset["test"].select(range(200))

# Label Extraction and Mapping (POS and Chunk Tags)

In [23]:
# Extract POS labels from ALL splits (train + validation + test)

pos_labels = sorted({
    tag
    for split in ["train", "validation", "test"]
    for ex in dataset[split]
    for tag in ex["pos_tags"]
})

# Create mapping (label → id and id → label)

pos_label2id = {l: i for i, l in enumerate(pos_labels)}
pos_id2label = {i: l for l, i in pos_label2id.items()}

# Extract Chunk labels from ALL splits
chunk_labels = sorted({
    tag
    for split in ["train", "validation", "test"]
    for ex in dataset[split]
    for tag in ex["chunk_tags"]
})

chunk_label2id = {l: i for i, l in enumerate(chunk_labels)}
chunk_id2label = {i: l for l, i in chunk_label2id.items()}

print("Labels Prepared Successfully")



Labels Prepared Successfully


# Tokenization and Label Alignment

In [24]:

from transformers import AutoTokenizer

# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Function to tokenize and align labels

def tokenize_align(examples, label_key, label2id):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    # Align labels with tokens
    for i, label in enumerate(examples[label_key]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # Ignore special tokens
            elif word_id != prev:
                label_ids.append(label2id[label[word_id]])  # Assign label
            else:
                label_ids.append(-100)  # Ignore subwords
            prev = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

# Apply tokenization
pos_data = dataset.map(lambda x: tokenize_align(x, "pos_tags", pos_label2id), batched=True)
chunk_data = dataset.map(lambda x: tokenize_align(x, "chunk_tags", chunk_label2id), batched=True)

print("Tokenization Done")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenization Done


# Model Setup using DistilBERT

In [25]:
from transformers import AutoModelForTokenClassification

# POS Model
pos_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(pos_labels),
    id2label=pos_id2label,
    label2id=pos_label2id
)

# Chunking Model
chunk_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_labels),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print("Models Loaded")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models Loaded


# Model Training using Hugging Face Trainer

In [26]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Data collator (handles padding automatically)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Load evaluation metric
metric = evaluate.load("seqeval")

# Compute evaluation metrics
def compute_metrics(label_list):
    def fn(p):
        preds, labels = p
        preds = np.argmax(preds, axis=2)

        true_preds = [
            [label_list[p] for p, l in zip(pred, lab) if l != -100]
            for pred, lab in zip(preds, labels)
        ]
        true_labels = [
            [label_list[l] for p, l in zip(pred, lab) if l != -100]
            for pred, lab in zip(preds, labels)
        ]

        res = metric.compute(predictions=true_preds, references=true_labels)

        return {
            "precision": res["overall_precision"],
            "recall": res["overall_recall"],
            "f1": res["overall_f1"]
        }
    return fn

# Training arguments (optimized for speed)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=50,
    report_to="none"
)

# Model Training and Evaluation

In [28]:
# POS Training
pos_trainer = Trainer(
    model=pos_model,
    args=training_args,
    train_dataset=pos_data["train"],
    eval_dataset=pos_data["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics(pos_labels)
)

print("Training POS Model...")
pos_trainer.train()

# Chunking Training
chunk_trainer = Trainer(
    model=chunk_model,
    args=training_args,
    train_dataset=chunk_data["train"],
    eval_dataset=chunk_data["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics(chunk_labels)
)

print("Training Chunk Model...")
chunk_trainer.train()


Training POS Model...


Step,Training Loss
50,0.733495
100,0.489356
150,0.411558
200,0.397989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Chunk Model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.121286
100,0.519987
150,0.413447
200,0.365424


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.6050358104705811, metrics={'train_runtime': 240.7646, 'train_samples_per_second': 3.323, 'train_steps_per_second': 0.831, 'total_flos': 8810722874208.0, 'train_loss': 0.6050358104705811, 'epoch': 1.0})

In [29]:
print("POS Evaluation:", pos_trainer.evaluate())
print("Chunk Evaluation:", chunk_trainer.evaluate())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarn

POS Evaluation: {'eval_loss': 0.36850807070732117, 'eval_precision': 0.8915929203539823, 'eval_recall': 0.8918395573997234, 'eval_f1': 0.8917162218227078, 'eval_runtime': 15.5776, 'eval_samples_per_second': 12.839, 'eval_steps_per_second': 1.605, 'epoch': 1.0}


Chunk Evaluation: {'eval_loss': 0.31309226155281067, 'eval_precision': 0.8368277119416591, 'eval_recall': 0.863593603010348, 'eval_f1': 0.85, 'eval_runtime': 15.058, 'eval_samples_per_second': 13.282, 'eval_steps_per_second': 1.66, 'epoch': 1.0}


# Inference on Custom Input Sentence

In [30]:
import torch

# Function for prediction
def predict(text, model, id2label):
    words = text.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0].tolist()
    word_ids = inputs.word_ids()

    result = []
    seen = set()

    for i, w_id in enumerate(word_ids):
        if w_id is None or w_id in seen:
            continue
        seen.add(w_id)
        result.append((words[w_id], id2label[preds[i]]))

    return result

# Test sentence
text = "John works at Google in California"

print("POS:", predict(text, pos_model, pos_id2label))
print("Chunk:", predict(text, chunk_model, chunk_id2label))

POS: [('John', 'NNP'), ('works', 'VBZ'), ('at', 'IN'), ('Google', 'NNP'), ('in', 'IN'), ('California', 'NNP')]
Chunk: [('John', 'B-NP'), ('works', 'B-VP'), ('at', 'B-PP'), ('Google', 'B-NP'), ('in', 'B-PP'), ('California', 'B-NP')]


# Comparison between POS Tagging and Chunking

In [34]:
# TASK 7: RESULT-BASED COMPARISON

import evaluate
import numpy as np

# Load metric again (fix error)
metric = evaluate.load("seqeval")

# Fix compute_metrics function
def compute_metrics(label_list):
    def fn(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [label_list[p] for p, l in zip(pred, lab) if l != -100]
            for pred, lab in zip(predictions, labels)
        ]

        true_labels = [
            [label_list[l] for p, l in zip(pred, lab) if l != -100]
            for pred, lab in zip(predictions, labels)
        ]

        results = metric.compute(
            predictions=true_predictions,
            references=true_labels
        )

        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"]
        }

    return fn


# Re-assign trainer with correct metric (IMPORTANT FIX)
pos_trainer.compute_metrics = compute_metrics(pos_labels)
chunk_trainer.compute_metrics = compute_metrics(chunk_labels)

# Run evaluation
pos_results = pos_trainer.evaluate()
chunk_results = chunk_trainer.evaluate()

# Print comparison
print("="*60)
print("        Experiment Comparison Summary")
print("="*60)

print(f"{'Metric':<15}{'POS Tagging':>15}{'Chunking':>15}")
print("-"*60)

metrics = ["eval_precision", "eval_recall", "eval_f1"]
labels  = ["Precision", "Recall", "F1 Score"]

for metric_name, label in zip(metrics, labels):
    pos_val = pos_results.get(metric_name, 0)
    chunk_val = chunk_results.get(metric_name, 0)
    print(f"{label:<15}{pos_val:>15.4f}{chunk_val:>15.4f}")

print("-"*60)
print("\nConclusion:")
print("POS Tagging achieved better performance than Chunking.")
print("Reason: POS Tagging is a simpler word-level task, making it easier for the model to learn patterns and achieve higher precision, recall, and F1 score compared to the more complex phrase-level chunking task.")


        Experiment Comparison Summary
Metric             POS Tagging       Chunking
------------------------------------------------------------
Precision               0.8916         0.8368
Recall                  0.8918         0.8636
F1 Score                0.8917         0.8500
------------------------------------------------------------

Conclusion:
POS Tagging achieved better performance than Chunking.
Reason: POS Tagging is a simpler word-level task, making it easier for the model to learn patterns and achieve higher precision, recall, and F1 score compared to the more complex phrase-level chunking task.


# Task 8: Report

**Differences between POS Tagging and Chunking**


***POS Tagging***



*   Works at word level
*   Assigns grammatical labels such as:

    1. Noun (NN)
    2. Verb (VB)
    3. Adjective (JJ)

*   Easier task because it focuses on individual words
*   Example:

      Sentence: "John works at Google"

      1. John → Noun
      2. works → Verb













 ***Chunking***



*  Works at phrase level

*  Groups words into meaningful phrases such as:
      1. Noun Phrase (NP)
      2. Verb Phrase (VP)
*   More complex because it requires understanding relationships between words
*  Example:

    Sentence: "John works at Google"

      1. John → B-NP
      2. works → B-VP




**Challenges Faced**

*   Handling tokenization and label alignment:

    1.   Transformer models split words into subwords
    2.   Used -100 for special tokens and subwords
  
* Dataset Handling Issues:

    1. Hugging Face dataset loading caused errors due to deprecated dataset scripts
    2. Used NLTK CoNLL-2000 dataset
    3. Converted it into Hugging Face format manually

*  Training Time Constraints:

    1. Large datasets increased training time
    2. Solution:
        Reduced dataset size for faster experimentation





 **Conclusion of Comparison**

*   POS Tagging achieved better performance than Chunking.
*   Reason: POS Tagging is a simpler word-level task, making it easier for the model to learn patterns and achieve higher precision, recall, and F1 score compared to the more complex phrase-level chunking task.


**Observations**

*  POS Tagging performed better than Chunking in all metrics.
*  It achieved higher precision, recall, and F1 score.
*  The model predicted POS tags more accurately and consistently.
*  Chunking performance was slightly lower, especially in precision.
*  Chunking was able to detect phrases but made more mistakes compared to POS tagging.

**Insights**

*  POS Tagging is easier because it works at the word level.
*  Chunking is more difficult because it works at the phrase level.
*  The model finds it easier to assign labels to individual words than to identify correct phrase boundaries.
*  Even small errors in chunking reduce overall performance.
*  Transformer models like DistilBERT still perform well on both tasks due to their ability to understand context.